# 21. Testing the too-many-parameters hypothesis

Dr Hunter, 5 November 2026, after seeing the 0.944 training against 0.644 test gap. He drew
a curve with several turning points and said you could either fit it exactly with a
high-order polynomial or approximate it with a straight line that has only two parameters:

> If you add more and more parameters, you can get something to fit any data, but whether or
> not that's meaningful is another matter. So could it be that the model that it's trying to
> fit has too many parameters and it tries to overfit the data? It's a possibility.

And at the end of the meeting:

> If you managed to get anything interesting out of trying the lighter-weight model, then let
> me know.

This notebook tests that hypothesis directly. It is a good hypothesis and it deserves a real
test rather than an opinion.

## How the hypothesis gets tested

If too much model capacity is causing the overfitting, then **reducing capacity should raise
the held-out score**. So reduce it in every way available and see whether anything improves:

| Part | What varies | Capacity goes down by |
|---|---|---|
| A | Regularisation strength `C` | shrinking the weights |
| B | Vocabulary size, via `min_df` and `max_features` | having fewer features |
| C | The model family itself | using a simpler learner |
| D | Nothing; reads existing results | comparing models already run |

Part D is worth doing first because it is free. The baseline has roughly 48,000 parameters
and XLM-RoBERTa has 279 million, a factor of about 6,000. If parameter count drove the
failure, those two should be far apart. They score 0.642 and 0.653.

**Runtime: about 4 minutes, CPU only.**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

# Try to locate the project directory, prioritizing Google Drive mount paths
possible_roots = [
    Path(os.environ.get('HINGLISH_HATE_ROOT', r'H:\Data\msc-hinglish-hate')),
    Path('/content/drive/MyDrive/msc-hinglish-hate'),
    Path('/content/drive/MyDrive/Project'),
    Path('/content/drive/MyDrive'),
    Path.cwd().parent
]

ROOT = None
for p in possible_roots:
    if (p / 'data').exists():
        ROOT = p
        break

if ROOT is None:
    # Fallback to current working directory parent if no matches found
    ROOT = Path.cwd().parent
    print(f"Warning: Could not find 'data' folder in default Google Drive paths. Defaulting ROOT to: {ROOT}")
else:
    print(f"Using ROOT path: {ROOT}")

sys.path.insert(0, str(ROOT / 'notebooks'))
DATA, W = ROOT / 'data', ROOT / 'writing'
FIG = W / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
SEED = 42

tr = pd.read_parquet('/content/drive/MyDrive/dissertation/data/splits/bohra_train.parquet')
te = pd.read_parquet('/content/drive/MyDrive/dissertation/data/splits/bohra_test.parquet')
Xtr, ytr = tr['text'], tr['label'].values
Xte, yte = te['text'], te['label'].values
print(f'train {len(ytr)} / test {len(yte)}')


def feats(min_df=2, max_features=None, word_ng=(1, 2), char_ng=(3, 5)):
    return FeatureUnion([
        ('word', TfidfVectorizer(ngram_range=word_ng, min_df=min_df,
                                 max_features=max_features, sublinear_tf=True)),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=char_ng, min_df=min_df,
                                 max_features=max_features, sublinear_tf=True)),
    ])


def evaluate(label, pipe, note=''):
    pipe.fit(Xtr, ytr)
    ptr, pte = pipe.predict(Xtr), pipe.predict(Xte)
    try:
        nfeat = len(pipe.named_steps['features'].get_feature_names_out())
    except Exception:
        nfeat = np.nan
    tracc, teacc = accuracy_score(ytr, ptr), accuracy_score(yte, pte)
    return {'setting': label, 'note': note, 'n_features': nfeat,
            'train_acc': round(tracc, 3), 'test_acc': round(teacc, 3),
            'gap': round(tracc - teacc, 3),
            'macro_f1': round(f1_score(yte, pte, average='macro'), 4),
            'hate_f1': round(f1_score(yte, pte, pos_label=1, zero_division=0), 3)}

train 3659 / test 915


## Part D first, because it costs nothing

Every model already run, ordered by how many parameters it has. If capacity were the problem,
this table would show a trend.

In [5]:
params = pd.DataFrame([
    {'model': 'Majority-class baseline',   'parameters': 1,           'macro_f1': 0.266},
    {'model': 'TF-IDF + LogReg',           'parameters': 47_734,      'macro_f1': 0.642},
    {'model': 'IndicBERT v1 (ALBERT)',     'parameters': 33_000_000,  'macro_f1': 0.652},
    {'model': 'MuRIL',                     'parameters': 238_000_000, 'macro_f1': 0.654},
    {'model': 'XLM-RoBERTa',               'parameters': 279_000_000, 'macro_f1': 0.653},
])
# recompute the majority-class figure rather than trusting a typed constant
dummy = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
params.loc[params.model.str.startswith('Majority'), 'macro_f1'] = round(
    f1_score(yte, dummy.predict(Xte), average='macro'), 3)

params['vs_baseline'] = (params['macro_f1'] - 0.642).round(3)
display(params)

span = params[params.parameters > 1]
print(f"\nParameter count spans {span.parameters.min():,} to {span.parameters.max():,}, "
      f"a factor of {span.parameters.max()/span.parameters.min():,.0f}.")
print(f"macro-F1 spans {span.macro_f1.min():.3f} to {span.macro_f1.max():.3f}, "
      f"a range of {span.macro_f1.max()-span.macro_f1.min():.3f}.")

,model,parameters,macro_f1,vs_baseline
0,Majority-class baseline,1,0.389,-0.253
1,TF-IDF + LogReg,47734,0.642,0.000
2,IndicBERT v1 (ALBERT),33000000,0.652,0.010
3,MuRIL,238000000,0.654,0.012
4,XLM-RoBERTa,279000000,0.653,0.011



Parameter count spans 47,734 to 279,000,000, a factor of 5,845.
macro-F1 spans 0.642 to 0.654, a range of 0.012.


## Part A. Regularisation

`C` is the inverse of regularisation strength in logistic regression. Small `C` means strong
regularisation, which shrinks the weights towards zero and lowers effective capacity. This is
the straight-line end of Hunter's drawing. Large `C` lets the model use its full capacity.

If his hypothesis holds, the held-out score should be highest at a small `C`.

In [6]:
rows = []
t0 = time.time()
for Cval in [0.001, 0.01, 0.05, 0.1, 0.3, 1.0, 3.0, 10.0, 100.0]:
    pipe = Pipeline([('features', feats()),
                     ('clf', LogisticRegression(max_iter=2000, C=Cval,
                                                class_weight='balanced',
                                                random_state=SEED))])
    r = evaluate(f'LogReg C={Cval}', pipe, note='regularisation sweep')
    r['C'] = Cval
    rows.append(r)
    print(f"  C={Cval:<7} features {r['n_features']:>6}  train {r['train_acc']:.3f}  "
          f"test {r['test_acc']:.3f}  gap {r['gap']:.3f}  macro-F1 {r['macro_f1']:.4f}")

reg = pd.DataFrame(rows)
best = reg.loc[reg.macro_f1.idxmax()]
print(f"\nbest: C={best.C}, macro-F1 {best.macro_f1:.4f}  (current setting is C=1.0)")
print(f'{time.time()-t0:.0f}s')

  C=0.001   features  47734  train 0.740  test 0.604  gap 0.135  macro-F1 0.5851
  C=0.01    features  47734  train 0.744  test 0.610  gap 0.134  macro-F1 0.5886
  C=0.05    features  47734  train 0.773  test 0.621  gap 0.152  macro-F1 0.6025
  C=0.1     features  47734  train 0.796  test 0.619  gap 0.178  macro-F1 0.6006
  C=0.3     features  47734  train 0.866  test 0.633  gap 0.234  macro-F1 0.6149
  C=1.0     features  47734  train 0.944  test 0.644  gap 0.301  macro-F1 0.6212
  C=3.0     features  47734  train 0.989  test 0.652  gap 0.337  macro-F1 0.6279
  C=10.0    features  47734  train 1.000  test 0.652  gap 0.347  macro-F1 0.6261
  C=100.0   features  47734  train 1.000  test 0.647  gap 0.353  macro-F1 0.6180

best: C=3.0, macro-F1 0.6279  (current setting is C=1.0)
14s


## Part B. Fewer features

The other way to cut capacity is to give the model less to work with. `min_df` drops terms
that appear in fewer than N documents; `max_features` keeps only the most frequent.

The current setting is `min_df=2`, which gives about 48,000 features for 3,659 training
examples: roughly thirteen features per example. That ratio is exactly what Hunter's
polynomial analogy is about, so this is the most direct test of it.

In [7]:
rows = []
for mdf in [2, 3, 5, 10, 20, 50, 100]:
    pipe = Pipeline([('features', feats(min_df=mdf)),
                     ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                                random_state=SEED))])
    r = evaluate(f'min_df={mdf}', pipe, note='vocabulary size sweep')
    r['min_df'] = mdf
    rows.append(r)
    print(f"  min_df={mdf:<4} features {r['n_features']:>6}  train {r['train_acc']:.3f}  "
          f"test {r['test_acc']:.3f}  gap {r['gap']:.3f}  macro-F1 {r['macro_f1']:.4f}")

for mf in [500, 1000, 2000, 5000, 10000]:
    pipe = Pipeline([('features', feats(max_features=mf)),
                     ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                                random_state=SEED))])
    r = evaluate(f'max_features={mf}', pipe, note='vocabulary size sweep')
    r['max_features'] = mf
    rows.append(r)
    print(f"  max_feat={mf:<6} features {r['n_features']:>6}  train {r['train_acc']:.3f}  "
          f"test {r['test_acc']:.3f}  gap {r['gap']:.3f}  macro-F1 {r['macro_f1']:.4f}")

vocab = pd.DataFrame(rows)
b = vocab.loc[vocab.macro_f1.idxmax()]
print(f"\nbest: {b.setting}, {int(b.n_features)} features, macro-F1 {b.macro_f1:.4f}")

  min_df=2    features  47734  train 0.944  test 0.644  gap 0.301  macro-F1 0.6212
  min_df=3    features  30772  train 0.926  test 0.647  gap 0.279  macro-F1 0.6241
  min_df=5    features  19004  train 0.899  test 0.640  gap 0.258  macro-F1 0.6192
  min_df=10   features  10175  train 0.859  test 0.637  gap 0.222  macro-F1 0.6172
  min_df=20   features   5466  train 0.807  test 0.639  gap 0.168  macro-F1 0.6203
  min_df=50   features   2295  train 0.756  test 0.627  gap 0.128  macro-F1 0.6116
  min_df=100  features   1163  train 0.725  test 0.627  gap 0.098  macro-F1 0.6120
  max_feat=500    features   1000  train 0.752  test 0.652  gap 0.099  macro-F1 0.6348
  max_feat=1000   features   2000  train 0.802  test 0.646  gap 0.156  macro-F1 0.6268
  max_feat=2000   features   4000  train 0.837  test 0.637  gap 0.200  macro-F1 0.6172
  max_feat=5000   features  10000  train 0.886  test 0.655  gap 0.231  macro-F1 0.6337
  max_feat=10000  features  20000  train 0.920  test 0.652  gap 0.268  

## Part C. Genuinely lighter-weight learners

Hunter's phrase was "the lighter-weight model". These are different model families, not just
a regularised logistic regression: a shallow decision tree has very few effective parameters,
and Naive Bayes has one weight per feature with no fitting of interactions at all.

In [8]:
rows = []
models = [
    ('Multinomial Naive Bayes',      MultinomialNB(alpha=1.0), {}),
    ('Decision tree, depth 5',       DecisionTreeClassifier(max_depth=5,
                                        class_weight='balanced', random_state=SEED), {}),
    ('Decision tree, depth 20',      DecisionTreeClassifier(max_depth=20,
                                        class_weight='balanced', random_state=SEED), {}),
    ('Random forest, 200 trees',     RandomForestClassifier(n_estimators=200, n_jobs=-1,
                                        class_weight='balanced', random_state=SEED), {}),
    ('Linear SVM',                   LinearSVC(class_weight='balanced',
                                        random_state=SEED), {}),
    ('LogReg, 1000 features',        LogisticRegression(max_iter=2000,
                                        class_weight='balanced', random_state=SEED),
                                     {'max_features': 1000}),
    ('LogReg, word unigrams only',   LogisticRegression(max_iter=2000,
                                        class_weight='balanced', random_state=SEED),
                                     {'word_ng': (1, 1), 'char_ng': (4, 4), 'min_df': 5}),
]
for name, clf, fkw in models:
    pipe = Pipeline([('features', feats(**fkw)), ('clf', clf)])
    r = evaluate(name, pipe, note='alternative learner')
    rows.append(r)
    print(f"  {name:28s} features {r['n_features']:>6}  train {r['train_acc']:.3f}  "
          f"test {r['test_acc']:.3f}  gap {r['gap']:.3f}  macro-F1 {r['macro_f1']:.4f}")

alts = pd.DataFrame(rows)

  Multinomial Naive Bayes      features  47734  train 0.752  test 0.669  gap 0.083  macro-F1 0.4959
  Decision tree, depth 5       features  47734  train 0.699  test 0.690  gap 0.009  macro-F1 0.6353
  Decision tree, depth 20      features  47734  train 0.896  test 0.595  gap 0.302  macro-F1 0.5767
  Random forest, 200 trees     features  47734  train 1.000  test 0.691  gap 0.309  macro-F1 0.6176
  Linear SVM                   features  47734  train 1.000  test 0.644  gap 0.356  macro-F1 0.6142
  LogReg, 1000 features        features   2000  train 0.802  test 0.646  gap 0.156  macro-F1 0.6268
  LogReg, word unigrams only   features   8190  train 0.881  test 0.648  gap 0.232  macro-F1 0.6263


In [9]:
allres = pd.concat([reg, vocab, alts], ignore_index=True)
allres.to_csv(W / 'capacity_sweep.csv', index=False)

current = 0.6213   # the C=1, min_df=2 configuration, no lexicon features
beat = allres[allres.macro_f1 > current + 0.005].sort_values('macro_f1', ascending=False)

print(f'current configuration: macro-F1 {current:.4f}')
print(f'configurations beating it by more than 0.005: {len(beat)}\n')
if len(beat):
    display(beat[['setting', 'n_features', 'train_acc', 'test_acc', 'gap', 'macro_f1']].head(10))
else:
    print('None. Reducing capacity did not improve held-out performance.')

print(f'\nbest overall: {allres.loc[allres.macro_f1.idxmax(), "setting"]} '
      f'at {allres.macro_f1.max():.4f}')
print(f'worst overall: {allres.loc[allres.macro_f1.idxmin(), "setting"]} '
      f'at {allres.macro_f1.min():.4f}')

current configuration: macro-F1 0.6213
configurations beating it by more than 0.005: 7



,setting,n_features,train_acc,test_acc,gap,macro_f1
22,"Decision tree, depth 5",47734,0.699,0.690,0.009,0.6353
16,max_features=500,1000,0.752,0.652,0.099,0.6348
19,max_features=5000,10000,0.886,0.655,0.231,0.6337
20,max_features=10000,20000,0.920,0.652,0.268,0.6297
6,LogReg C=3.0,47734,0.989,0.652,0.337,0.6279
17,max_features=1000,2000,0.802,0.646,0.156,0.6268
26,"LogReg, 1000 features",2000,0.802,0.646,0.156,0.6268



best overall: Decision tree, depth 5 at 0.6353
worst overall: Multinomial Naive Bayes at 0.4959


In [10]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

# A: regularisation
axes[0].semilogx(reg['C'], reg['macro_f1'], 'o-', color='#2b7bba', label='test macro-F1')
axes[0].semilogx(reg['C'], reg['train_acc'], 'o--', color='#9ecae1', label='train accuracy')
axes[0].semilogx(reg['C'], reg['test_acc'], 'o-', color='#d1495b', label='test accuracy')
axes[0].set_xlabel('C  (left = simpler model)'); axes[0].set_ylim(0, 1.02)
axes[0].set_title('A. Regularisation', fontsize=10)
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.25)

# B: vocabulary size
v = vocab.sort_values('n_features')
axes[1].semilogx(v['n_features'], v['macro_f1'], 'o-', color='#2b7bba', label='test macro-F1')
axes[1].semilogx(v['n_features'], v['train_acc'], 'o--', color='#9ecae1', label='train accuracy')
axes[1].set_xlabel('number of features'); axes[1].set_ylim(0, 1.02)
axes[1].set_title('B. Vocabulary size', fontsize=10)
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.25)

# D: parameters across every model run
p = params[params.parameters > 1]
axes[2].semilogx(p['parameters'], p['macro_f1'], 'o', markersize=9, color='#d1495b')
for _, r in p.iterrows():
    axes[2].annotate(r['model'].split(' (')[0], (r['parameters'], r['macro_f1']),
                     fontsize=7, xytext=(0, 8), textcoords='offset points', ha='center')
axes[2].set_xlabel('parameters'); axes[2].set_ylim(0.55, 0.72)
axes[2].set_title('D. Capacity across model families', fontsize=10)
axes[2].grid(alpha=0.25)

fig.suptitle('Does reducing model capacity help?', fontsize=11)
fig.tight_layout()
fig.savefig(FIG / 'capacity_sweep.png', dpi=200, bbox_inches='tight'); plt.close(fig)
print('saved capacity_sweep.png')

saved capacity_sweep.png


## The check that decides it: cross-validation

The sweep above picked a winner from a single 915-post test split after trying more than
twenty configurations. That is exactly the situation where a result can look good by chance,
because searching many settings against one test set is a way of overfitting the test set
rather than the training set.

So the leading candidates are re-run under five-fold cross-validation on all 4,574 posts.
If an improvement is real it survives; if it was noise it disappears.

**Read this table rather than the one above.**

In [11]:
from sklearn.model_selection import StratifiedKFold

Xall = pd.concat([tr['text'], te['text']]).reset_index(drop=True)
yall = np.concatenate([tr['label'].values, te['label'].values])
print(f'pooled {len(yall)} posts for 5-fold CV\n')

CANDIDATES = [
    ('current (C=1, min_df=2)', lambda: Pipeline([('features', feats()),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))])),
    ('max_features=500',        lambda: Pipeline([('features', feats(max_features=500)),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))])),
    ('max_features=5000',       lambda: Pipeline([('features', feats(max_features=5000)),
        ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))])),
    ('C=3.0',                   lambda: Pipeline([('features', feats()),
        ('clf', LogisticRegression(max_iter=2000, C=3.0, class_weight='balanced', random_state=SEED))])),
    ('Decision tree, depth 5',  lambda: Pipeline([('features', feats()),
        ('clf', DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=SEED))])),
]

rows = []
for name, factory in CANDIDATES:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    mf, hf = [], []
    for i_tr, i_te in skf.split(Xall, yall):
        pp = factory(); pp.fit(Xall.iloc[i_tr], yall[i_tr])
        pred = pp.predict(Xall.iloc[i_te])
        mf.append(f1_score(yall[i_te], pred, average='macro'))
        hf.append(f1_score(yall[i_te], pred, pos_label=1, zero_division=0))
    rows.append({'config': name,
                 'macro_f1_mean': round(np.mean(mf), 4), 'macro_f1_sd': round(np.std(mf), 4),
                 'hate_f1_mean': round(np.mean(hf), 4),  'hate_f1_sd': round(np.std(hf), 4)})
    print(f'  {name:26s} macro-F1 {np.mean(mf):.4f} +/- {np.std(mf):.4f}   '
          f'hate-F1 {np.mean(hf):.4f} +/- {np.std(hf):.4f}')

cv = pd.DataFrame(rows)
cv.to_csv(W / 'capacity_sweep_cv.csv', index=False)
base = cv.iloc[0]
print(f"\nagainst the current configuration ({base.macro_f1_mean:.4f} "
      f"+/- {base.macro_f1_sd:.4f}):")
for _, x in cv.iloc[1:].iterrows():
    d = x.macro_f1_mean - base.macro_f1_mean
    verdict = 'outside' if abs(d) > 2 * base.macro_f1_sd else 'INSIDE'
    print(f"  {x.config:26s} {d:+.4f}   {verdict} two standard deviations")

pooled 4574 posts for 5-fold CV

  current (C=1, min_df=2)    macro-F1 0.6473 +/- 0.0136   hate-F1 0.5609 +/- 0.0144
  max_features=500           macro-F1 0.6316 +/- 0.0208   hate-F1 0.5499 +/- 0.0233
  max_features=5000          macro-F1 0.6476 +/- 0.0130   hate-F1 0.5611 +/- 0.0159
  C=3.0                      macro-F1 0.6467 +/- 0.0112   hate-F1 0.5536 +/- 0.0166
  Decision tree, depth 5     macro-F1 0.6141 +/- 0.0095   hate-F1 0.5235 +/- 0.0365

against the current configuration (0.6473 +/- 0.0136):
  max_features=500           -0.0157   INSIDE two standard deviations
  max_features=5000          +0.0003   INSIDE two standard deviations
  C=3.0                      -0.0006   INSIDE two standard deviations
  Decision tree, depth 5     -0.0332   outside two standard deviations


## Part E. Confusion matrices for the new experiments

He asked for these again at the end of the meeting. The existing ones in notebook 14 cover
the Hinglish models; these cover the two experiments he saw for the first time today.

In [14]:
import sys
from pathlib import Path

# Manually define the correct Google Drive path for DATA and dissertation scripts
DATA = Path('/content/drive/MyDrive/dissertation/data')
sys.path.insert(0, '/content/drive/MyDrive/dissertation/notebooks')
sys.path.insert(0, '/content/drive/MyDrive/dissertation')

from sklearn.model_selection import train_test_split
from hinglish_hate import load_hasoc2022_threads, filter_romanised

LAB = ['not hate', 'hate']

def draw(ax, y, p, title):
    cm = confusion_matrix(y, p, labels=[0, 1])
    row = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    ax.imshow(row, cmap='Blues', vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i,j]}\n{row[i,j]*100:.1f}%', ha='center', va='center',
                    fontsize=9, color='white' if row[i, j] > 0.5 else 'black')
    ax.set_xticks([0, 1]); ax.set_xticklabels(LAB, fontsize=8)
    ax.set_yticks([0, 1]); ax.set_yticklabels(LAB, fontsize=8, rotation=90, va='center')
    ax.set_xlabel('predicted', fontsize=8); ax.set_ylabel('actual', fontsize=8)
    ax.set_title(title, fontsize=9)

panels = []

# English, matched and full
CSV = DATA / 'english' / 'davidson_labeled_data.csv'
if CSV.exists():
    eng = pd.read_csv(CSV)
    eng['label'] = (eng['class'] == 0).astype(int)
    eng = eng[['tweet', 'label']].rename(columns={'tweet': 'text'}).dropna()
    hate, notu = eng[eng.label == 1], eng[eng.label == 0]
    m = pd.concat([hate, notu.sample(int(round(len(hate)*(1-0.363)/0.363)),
                                     random_state=SEED)]).sample(frac=1, random_state=SEED)
    mtr, mte = train_test_split(m, test_size=0.2, stratify=m['label'], random_state=SEED)
    pipe = Pipeline([('features', feats()),
                     ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                                random_state=SEED))])
    pipe.fit(mtr['text'], mtr['label'])
    panels.append((mte['label'].values, pipe.predict(mte['text']), 'English, matched'))
else:
    print('English data missing; run notebook 17 first.')

# Hinglish reference with the same lexicon-free pipeline
pipe = Pipeline([('features', feats()),
                 ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                            random_state=SEED))])
pipe.fit(Xtr, ytr)
panels.insert(0, (yte, pipe.predict(Xte), 'Hinglish (Bohra)'))

# within-HASOC and both cross directions
h22 = filter_romanised(load_hasoc2022_threads(DATA), include_mixed=True).reset_index(drop=True)
tr22, te22 = train_test_split(h22, test_size=0.2, stratify=h22['label'], random_state=SEED)
p22 = Pipeline([('features', feats()),
                ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                           random_state=SEED))])
p22.fit(tr22['text'], tr22['label'])
panels.append((te22['label'].values, p22.predict(te22['text']), 'HASOC -> HASOC'))
panels.append((yte, p22.predict(Xte), 'HASOC -> Bohra'))
panels.append((te22['label'].values, pipe.predict(te22['text']), 'Bohra -> HASOC'))

fig, axes = plt.subplots(1, len(panels), figsize=(3.1*len(panels), 3.3))
for ax, (y, p, t) in zip(np.atleast_1d(axes), panels):
    draw(ax, y, p, t)
fig.suptitle('Confusion matrices: English control and within-HASOC', fontsize=11)
fig.tight_layout()
fig.savefig(FIG / 'confusion_new_experiments.png', dpi=200, bbox_inches='tight'); plt.close(fig)
print('saved confusion_new_experiments.png')

English data missing; run notebook 17 first.
saved confusion_new_experiments.png


## What to tell him

The result decides the wording, so read the output before writing anything, but it will fall
into one of two cases.

**If nothing beat the current configuration.** Say it plainly: the hypothesis was tested and
it does not hold. Reducing capacity by regularisation, by cutting the vocabulary from 48,000
features down to a few hundred, and by switching to simpler learners entirely, none of it
raised the held-out score. Add the free argument from Part D: the models already run span
from about 48,000 parameters to 279 million, a factor of roughly 6,000, and they all land
within 0.012 macro-F1 of each other. Capacity is not what separates them.

That is a genuinely useful negative result, and it is the kind of thing that reads well in a
dissertation because it shows a hypothesis being tested rather than asserted.

**If something did beat it.** Report the winning configuration, say by how much, and be
careful not to overclaim a small gain found by searching many settings. If the improvement is
under about 0.01 it is within the seed-to-seed spread you already measured, so say so.

Either way the experiment was worth running, and it was his idea. Tell him that.